# Validate linear summation (metamodel vs dynamic)

Compare the offshore-driven **dynamic SnapWave** run (08, no wind) with the
**metamodel reconstruction** (linear sum of the goal spectra) at every output
site of every metamodel set, in both modes:

- **bulk**: one WHACS bulk spectrum per goal;
- **spectral**: WHACS partitions `0`–`3` per goal.

Both read the same water level as the dynamic run (`wl="forcing"`) and come
back with the depth cap (`metamodel.reconstruction.depth_cap`). Section 5 shows
what the cap changes. Bulk metrics per set are saved for the portal.

Spread: the dynamic run clips the boundary `spr` at 55° (SnapWave, notebook 08),
the reconstruction does not (bulk mode above the training bounds extrapolates; the
spectral partitions sit inside them). Where the offshore spread is broad the dynamic
run is not a clean reference: judge those hours at the buoys (10). `tm01` exists only
in the reconstructions (a stat of the summed spectrum), so it is compared at the buoys,
not here.

**Prerequisites:** fitted models (06), dynamic run postprocessed (08,
`Dynamic_Cases_<period>.nc`). The reconstructions are read from the cache
(`tmp/reconstruct/`); anything missing is computed here.

In [ ]:
import time

import matplotlib.pyplot as plt
import pandas as pd
import xarray as xr
from utils import metamodel as mm
from utils import simulation as sim
from utils import validation as val

## Configuration

In [ ]:
period = "2018"  # TODO: the year of the dynamic run (08)
offshore_name = "WHACS"  # TODO: offshore_waves entry of the dynamic run
wind = False  # dynamic run with or without wind (08); the metamodel has none
mda_n = 600  # must match a fitted mda_candidates entry
sets = list(mm.SETS)  # metamodel sets to validate
modes = {"bulk": False, "spectral": True}
wl = "forcing"  # same water level as the dynamic run

compare_variables = ["hs", "tp", "dir", "spr"]

dynamic_path = (
    sim.cases_dir("dynamic", offshore=offshore_name, wind=wind)
    / f"Dynamic_Cases_{period}.nc"
)

print(f"period       : {period}")
print(f"offshore     : {offshore_name}")
print(f"mda_n        : {mda_n}")
print(f"sets         : {sets}")
print(f"dynamic path : {dynamic_path}")

## 1 · Check models

In [ ]:
for mm_set in sets:
    mm.require_models(mm_set, mda_n)
    print(f"{mm_set:10s}: {len(mm.reconstruction_sites(mm_set))} sites kept in every goal")

## 2 · Reconstruct (every set, bulk and spectral)

In [ ]:
# Resources for the reconstructions below (at most 75% of this machine),
# sized on the largest call. Edit res if wanted, then run the next cell.
res = mm.reconstruction_resources(
    period, mda_n, metamodel="isobaths", spectral=True,
    offshore_name=offshore_name, save_spectra=False, wl=wl,
)

In [ ]:
recon: dict[tuple[str, str], xr.Dataset] = {}
for mm_set in sets:
    for mode, spectral in modes.items():
        t0 = time.perf_counter()
        recon[mm_set, mode] = mm.reconstruct(
            period, mda_n, metamodel=mm_set, spectral=spectral,
            offshore_name=offshore_name, resources=res, save_spectra=False, wl=wl,
        )
        print(f"{mm_set:10s} {mode:8s}: {recon[mm_set, mode].sizes['sites']} sites, "
              f"{time.perf_counter() - t0:.1f} s")

## 3 · Dynamic reference

In [ ]:
dynamic = xr.open_dataset(dynamic_path).sel(time=period).load()
print(f"{dynamic.sizes['sites']} sites x {dynamic.sizes['time']} hours")
print({v: f"{float(dynamic[v].isnull().mean()):.4f}" for v in compare_variables if v in dynamic}, "NaN fraction")

## 4 · Compare

Metrics per site group (isobath, polygon, buoy), reference = dynamic. Table:
median over the groups of each set.

In [ ]:
results: dict[tuple[str, str], val.Comparison] = {}
for (mm_set, mode), est in recon.items():
    variables = val.resolve_comparison_variables(est, dynamic, variables=compare_variables)
    results[mm_set, mode] = val.compare(
        reference=dynamic, estimate=est, variables=variables,
        reference_label=val.DYNAMIC_REFERENCE_LABEL,
        estimate_label=f"Linear sum ({mode})",
    )

metrics = pd.concat(
    [r.metrics.assign(set=s, mode=m) for (s, m), r in results.items()], ignore_index=True
)
metrics.pivot_table(
    index=["variable", "set"], columns="mode", values=["bias", "rmse", "r2"], aggfunc="median"
).round(3)

In [ ]:
# Every group, Hs
metrics[metrics.variable == "hs"].pivot_table(
    index=["set", "group"], columns="mode", values=["bias", "rmse", "r2"]
).round(3)

### Save bulk metrics (one file per set, read by `publish`)

In [ ]:
for mm_set in sets:
    metrics_out = results[mm_set, "bulk"].metrics.copy()
    metrics_out.insert(0, "period", period)
    metrics_out.insert(1, "metamodel", mm_set)
    metrics_out.insert(2, "mda_candidate", mda_n)
    metrics_out.insert(3, "offshore_name", offshore_name)
    metrics_out["n_sites"] = int(recon[mm_set, "bulk"].sizes["sites"])
    out_path = val.metamodel_validation_path(period, mm_set)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    metrics_out.to_csv(out_path, index=False)
    print(f"Wrote {out_path}")

## 5 · Depth cap effect

The same bulk reconstructions without the cap (`depth_cap=False`, read from the
same cache): Hs metrics per group, capped vs plain sum.

In [ ]:
rows = []
for mm_set in sets:
    plain = mm.reconstruct(
        period, mda_n, metamodel=mm_set, spectral=False, offshore_name=offshore_name,
        resources=res, save_spectra=False, wl=wl, depth_cap=False,
    )
    for cap, est in (("plain sum", plain), ("capped", recon[mm_set, "bulk"])):
        m = val.compare(reference=dynamic, estimate=est, variables=["hs"]).metrics
        rows.append(m.assign(set=mm_set, depth_cap=cap))
    changed = float((plain.hs > recon[mm_set, "bulk"].hs + 1e-6).mean())
    print(f"{mm_set:10s}: {100 * changed:.2f} % of site-hours lowered by the cap")
pd.concat(rows).pivot_table(
    index=["set", "group"], columns="depth_cap", values=["bias", "rmse", "r2"]
).round(3)

## 6 · Plots per site group (bulk, then spectral)

In [ ]:
for (mm_set, mode), result in results.items():
    for fig in result.plot_all(variables=compare_variables):
        fig.suptitle(f"{mm_set} · {mode} — {period}", y=1.02)
        plt.show()

## 7 · Time series at the buoys (bulk and spectral vs dynamic)

In [ ]:
buoy_estimates = {f"Linear sum ({mode})": recon["buoys", mode] for mode in modes}
for site in recon["buoys", "bulk"].sites.values:
    val.plot_buoy_timeseries(
        dynamic, buoy_estimates, site, variables=compare_variables,
        reference_label=val.DYNAMIC_REFERENCE_LABEL,
    )
    plt.suptitle(f"Dynamic vs linear sum — {site} — {period}")
    plt.show()